In [ ]:
import os
import time
import torch
import numpy as np
import gymnasium as gym
import pybullet as p
import pybullet_data
import matplotlib.pyplot as plt
import seaborn as sns
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

# --- 0. FORCE RESET KERNEL LOGIC ---
# If you see "Only one local connection" again, please click 
# the "RESTART" button at the top of your VS Code window.
try:
    p.disconnect()
except:
    pass

# CONNECT ONCE - Keep this open for the whole session
p.connect(p.GUI) 
p.configureDebugVisualizer(p.COV_ENABLE_GUI, 0)
p.setAdditionalSearchPath(pybullet_data.getDataPath())

sns.set_theme(style="darkgrid")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. 3D ENVIRONMENT ---
class SafeNav3DEnv(gym.Env):
    def __init__(self, size=10, max_hazards=5, curriculum=False):
        super().__init__()
        self.size = size
        self.max_hazards = max_hazards
        self.curriculum = curriculum
        self.episode_count = 0
        self.action_space = gym.spaces.Discrete(4)
        self.observation_space = gym.spaces.Box(low=-20, high=20, shape=(9 + 3*max_hazards,), dtype=np.float32)
        self.goal_pos = np.array([size-1, size-1, 0.5])

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        p.resetSimulation()
        p.setGravity(0, 0, -9.81)
        p.loadURDF("plane.urdf")
        self.agent_id = p.loadURDF("sphere2.urdf", [0, 0, 0.5], globalScaling=0.5)
        p.changeVisualShape(self.agent_id, -1, rgbaColor=[0, 0, 1, 1])
        goal_visual = p.createVisualShape(p.GEOM_SPHERE, radius=0.6, rgbaColor=[0, 1, 0, 0.5])
        p.createMultiBody(baseVisualShapeIndex=goal_visual, basePosition=self.goal_pos)
        self.episode_count += 1
        self._setup_hazards()
        return self._get_obs(), {}

    def _setup_hazards(self):
        num_hazards = min(self.max_hazards, 1 + self.episode_count // 50) if self.curriculum else self.max_hazards
        self.hazard_positions = []
        for _ in range(num_hazards):
            h_pos = [np.random.uniform(-self.size, self.size), np.random.uniform(-self.size, self.size), 0.5]
            p.loadURDF("r2d2.urdf", h_pos, globalScaling=0.6)
            self.hazard_positions.append(h_pos)

    def _get_obs(self):
        pos, _ = p.getBasePositionAndOrientation(self.agent_id)
        vel, _ = p.getBaseVelocity(self.agent_id)
        obs = list(pos) + list(vel) + list(self.goal_pos)
        for h in self.hazard_positions: obs.extend(h)
        while len(obs) < self.observation_space.shape[0]: obs.extend([0, 0, 0])
        return np.array(obs, dtype=np.float32)

    def step(self, action):
        force = [0, 0, 0]
        mag = 12
        if action == 0: force[1] = mag
        elif action == 1: force[1] = -mag
        elif action == 2: force[0] = -mag
        elif action == 3: force[0] = mag
        p.applyExternalForce(self.agent_id, -1, force, [0, 0, 0], p.WORLD_FRAME)
        p.stepSimulation()
        obs = self._get_obs()
        reward, cost, done = -0.1, 0, False
        if np.linalg.norm(obs[0:3] - self.goal_pos) < 1.0:
            reward += 100
            done = True
        for h_pos in self.hazard_positions:
            if np.linalg.norm(obs[0:3] - np.array(h_pos)) < 1.3:
                cost, reward, done = 1, -50, True
        return obs, reward, done, False, {'cost': cost}

# --- 2. SAFETY LOGIC ---
class SafetyShield:
    def check_and_fix(self, obs, action):
        pos = obs[0:3]
        for i in range(9, len(obs), 3):
            h_pos = obs[i:i+3]
            if np.all(h_pos == 0): continue
            if np.linalg.norm(pos - h_pos) < 2.2:
                return np.random.choice([0,1,2,3]), True
        return action, False

class ShieldedEnv(gym.Wrapper):
    def __init__(self, env, shield):
        super().__init__(env)
        self.shield = shield
        self.interventions = 0
    def step(self, action):
        obs = self.env._get_obs()
        safe_action, intervened = self.shield.check_and_fix(obs, action)
        if intervened: self.interventions += 1
        return self.env.step(safe_action)

# --- 3. METRICS CALLBACK ---
class MetricsCallback(BaseCallback):
    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.rewards, self.costs, self.interventions = [], [], []
    def _on_step(self) -> bool:
        if 'episode' in self.locals['infos'][0]:
            info = self.locals['infos'][0]
            self.rewards.append(info['episode']['r'])
            self.costs.append(info.get('cost', 0))
            # Accessing from the single env
            root_env = self.training_env.envs[0]
            self.interventions.append(root_env.interventions if hasattr(root_env, 'interventions') else 0)
        return True

# --- 4. TRAINING ---
print("🚀 Training 3D SafeRL Agent...")
base_env = SafeNav3DEnv(curriculum=True)
shield_env = ShieldedEnv(base_env, SafetyShield())
monitored_env = Monitor(shield_env)
v_env = DummyVecEnv([lambda: monitored_env])

model = PPO("MlpPolicy", v_env, verbose=1, device=device)
cb = MetricsCallback()

# Lower timesteps to 15000 for a quick test to ensure graphs populate
model.learn(total_timesteps=15000, callback=cb)

# --- 5. VISUALIZATION ---
print("📊 Generating Visualization Graphs...")
if len(cb.rewards) > 0:
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))
    sns.lineplot(data=cb.rewards, ax=ax[0], color='blue').set_title("Rewards")
    sns.lineplot(data=np.cumsum(cb.costs), ax=ax[1], color='red').set_title("Total Crashes")
    sns.lineplot(data=cb.interventions, ax=ax[2], color='green').set_title("Interventions")
    plt.tight_layout()
    plt.show()
else:
    print("❌ Graphs are empty because training didn't complete any episodes.")

# --- 6. DEMO ---
print("📺 Final 3D Visual Demonstration...")
obs, _ = base_env.reset()
for _ in range(1000):
    action, _ = model.predict(obs, deterministic=True)
    obs, r, done, _, _ = base_env.step(action)
    time.sleep(1./60.)
    if done: obs, _ = base_env.reset()

p.disconnect()